# PRCL: Poison-Robust Contrastive Learning — Colab Training Notebook

This notebook provides a portable, end-to-end pipeline for running PRCL experiments on Google Colab (or any Jupyter environment with GPU).

**Contents:**
1. Setup & Installation
2. Configuration
3. Clean Baseline Training
4. Poisoned Training (Undefended)
5. PRCL-Defended Training
6. Evaluation & Visualization

> **Runtime:** Select *Runtime → Change runtime type → T4 GPU* (or better) before running.

## 1. Setup & Installation

In [ ]:
# Clone the repository
!git clone https://github.com/aayushakumar/PRCL.git
%cd PRCL

In [ ]:
# Install dependencies (editable mode for dev access)
!pip install -e '.[dev]' -q

In [ ]:
# Verify GPU availability
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Quick sanity check — run tests
!python -m pytest tests/ -x -q --tb=short 2>&1 | tail -5

## 2. Configuration

Adjust these parameters to control the experiment. For quick smoke tests, use low epoch counts and a small subset.

In [ ]:
# ── Experiment parameters (edit these) ──────────────────────────
DATASET = "cifar10"            # cifar10 | cifar100 | stl10
BACKBONE = "resnet18"          # resnet18 | resnet50
SEED = 42

# Training
SSL_EPOCHS = 50                # 200 for full run, 50 for quick test
SSL_BATCH_SIZE = 256
SSL_LR = 3e-4

# Attack
ATTACK_TYPE = "patch_backdoor"  # patch_backdoor | blend_backdoor | none
POISON_RATIO = 0.01            # fraction of training set to poison
TARGET_CLASS = 0

# Evaluation
EVAL_EPOCHS = 100
EVAL_LR = 0.1

# Subset for smoke testing (set to None for full dataset)
SUBSET_SIZE = None              # e.g., 5000 for quick test

# Output
RUN_DIR = "./runs"
# ────────────────────────────────────────────────────────────────

In [ ]:
# Helper to build Hydra override strings from the parameters above
def _base_overrides():
    overrides = [
        f"dataset={DATASET}",
        f"model={BACKBONE}",
        f"seed={SEED}",
        f"ssl.epochs={SSL_EPOCHS}",
        f"ssl.batch_size={SSL_BATCH_SIZE}",
        f"ssl.lr={SSL_LR}",
        f"eval.epochs={EVAL_EPOCHS}",
        f"eval.lr={EVAL_LR}",
        f"run_dir={RUN_DIR}",
    ]
    if SUBSET_SIZE is not None:
        overrides.append(f"dataset.subset_size={SUBSET_SIZE}")
    return overrides


def run_train(extra_overrides=None):
    """Launch training via Hydra CLI."""
    parts = ["python", "scripts/train.py"] + _base_overrides()
    if extra_overrides:
        parts.extend(extra_overrides)
    cmd = " ".join(parts)
    print(f"▶ {cmd}\n")
    !{cmd}

## 3. Clean Baseline Training

Train SimCLR on the clean dataset (no attack, no defense) to establish the baseline clean accuracy.

In [ ]:
run_train([
    "attack=none",
    "defense=none",
])

## 4. Poisoned Training (Undefended)

Train on the poisoned dataset **without** any defense, to measure the attack success rate (ASR) ceiling.

In [ ]:
run_train([
    f"attack={ATTACK_TYPE}",
    f"attack.poison_ratio={POISON_RATIO}",
    f"attack.target_class={TARGET_CLASS}",
    "defense=none",
])

## 5. PRCL-Defended Training

Train on the same poisoned dataset, but with the full PRCL defense pipeline enabled (PCF scoring → positive reweighting → negative sanitization → gradient capping).

In [ ]:
run_train([
    f"attack={ATTACK_TYPE}",
    f"attack.poison_ratio={POISON_RATIO}",
    f"attack.target_class={TARGET_CLASS}",
    "defense=prcl",
])

## 6. Evaluation & Visualization

Load the run metrics and produce comparison plots.

In [ ]:
import json
from pathlib import Path


def load_run_metrics(run_dir: str) -> dict:
    """Scan run directory and return mapping of run_name → metrics."""
    results = {}
    runs_path = Path(run_dir)
    if not runs_path.exists():
        print(f"No runs found in {run_dir}")
        return results
    for run_folder in sorted(runs_path.iterdir()):
        metrics_file = run_folder / "metrics.json"
        if metrics_file.exists():
            with open(metrics_file) as f:
                results[run_folder.name] = json.load(f)
    return results


all_metrics = load_run_metrics(RUN_DIR)
for name, m in all_metrics.items():
    acc = m.get("clean_acc", m.get("test_acc", "N/A"))
    asr = m.get("asr", "N/A")
    print(f"{name}:  Clean Acc = {acc}  |  ASR = {asr}")

In [ ]:
import sys
sys.path.insert(0, "src")

import matplotlib.pyplot as plt
import numpy as np
from prcl.integritysuite.plots.visualizations import plot_asr_comparison

# ── ASR comparison bar chart ─────────────────────────────────
asr_data = {}
for name, m in all_metrics.items():
    if "asr" in m:
        # Shorten name for readability
        label = name.split("_seed")[0] if "_seed" in name else name
        asr_data[label] = m["asr"]

if asr_data:
    out = plot_asr_comparison(asr_data, save_path="asr_comparison.png")
    from IPython.display import Image, display
    display(Image(filename=str(out)))
else:
    print("No ASR data found — run poisoned experiments first.")

In [ ]:
from prcl.integritysuite.plots.visualizations import plot_training_curves

# ── Training curves for each run ─────────────────────────────
for name, m in all_metrics.items():
    epoch_metrics = m.get("epoch_metrics", [])
    if epoch_metrics:
        save_path = f"train_curve_{name}.png"
        plot_training_curves(epoch_metrics, save_path=save_path)
        from IPython.display import Image, display
        display(Image(filename=save_path))
        print(f"\n{name}")

In [ ]:
# ── Aggregate report (markdown tables) ───────────────────────
!python scripts/report.py --runs-dir {RUN_DIR}

## Appendix: Custom Sweep

Run a quick poison-ratio sweep from this notebook:

In [ ]:
# ── Poison-ratio sweep ───────────────────────────────────────
SWEEP_RATIOS = [0.005, 0.01, 0.03, 0.05, 0.10]

for ratio in SWEEP_RATIOS:
    print(f"\n{'='*60}")
    print(f"  Poison ratio: {ratio*100:.1f}%")
    print(f"{'='*60}")

    # Undefended
    run_train([
        f"attack={ATTACK_TYPE}",
        f"attack.poison_ratio={ratio}",
        "defense=none",
    ])

    # PRCL-defended
    run_train([
        f"attack={ATTACK_TYPE}",
        f"attack.poison_ratio={ratio}",
        "defense=prcl",
    ])

---

**Note:** For full reproduction of paper results, use `scripts/reproduce_main_tables.sh` on a multi-GPU machine. See [docs/reproducibility.md](../docs/reproducibility.md) for details.